In [1]:
from slurm_decode_base import *
import sys
import os

MODELS_PATH = '/home/users/bensonb/international-brain-lab/behavior_models'
if not MODELS_PATH in sys.path:                                                                              
     sys.path.insert(0, MODELS_PATH)
SCRATCH = os.environ['SCRATCH']

import pickle
import logging
import numpy as np
import pandas as pd
import decoding_utils as dut
import brainbox.io.one as bbone
import sklearn.linear_model as sklm
import models.utils as mut
from pathlib import Path
from datetime import date
from one.api import ONE
from models.expSmoothing_prevAction import expSmoothing_prevAction
from models.optimalBayesian import optimal_Bayesian
# from brainbox.singlecell import calculate_peths
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.task.closed_loop import generate_pseudo_session
from brainbox.metrics.single_units import quick_unit_metrics
from decoding_stimulus_neurometric_fit import get_neurometric_parameters

from tqdm import tqdm
from ibllib.atlas import AllenAtlas

logger = logging.getLogger('ibllib')
logger.disabled = True

strlut = {sklm.Lasso: 'Lasso',
          sklm.LassoCV: 'LassoCV',
          sklm.Ridge: 'Ridge',
          sklm.RidgeCV: 'RidgeCV',
          sklm.LinearRegression: 'PureLinear',
          sklm.LogisticRegression: 'Logistic'}

# %% Run param definitions

# aligned -> histology was performed by one experimenter
# resolved -> histology was performed by 2-3 experiments
SESS_CRITERION = 'aligned-behavior'  # aligned and behavior
MODEL = optimal_Bayesian  # None or expSmoothing_prevAction or dut.modeldispatcher
DATE = str(date.today())
MODELFIT_PATH = os.path.join(SCRATCH,'international-brain-lab/prior-localization/behavior/')
OUTPUT_PATH = os.path.join(SCRATCH,'international-brain-lab/prior-localization/decoding/')

#MODELFIT_PATH = '/Users/csmfindling/Documents/Postdoc-Geneva/IBL/behavior/prior-localization/decoding/results/behavior/'
#OUTPUT_PATH = '/Users/csmfindling/Documents/Postdoc-Geneva/IBL/behavior/prior-localization/decoding/results/decoding/'
ALIGN_TIME = 'goCue_times'# 'feedback_times'
TARGET = 'prior'  # 'pLeft','prior','choice','feedback','signcont'
TIME_WINDOW = (-0.6, -0.2)  # (-0.6, -0.2), (0, 0.1)
ESTIMATOR = sklm.Lasso #sklm.LogisticRegression #sklm.Lasso  # Must be in keys of strlut above
ESTIMATOR_KWARGS = {'tol': 0.0001, 'max_iter': 10000, 'fit_intercept': True}#'penalty': 'l1', 'solver':'saga', 
N_PSEUDO = 0
MIN_UNITS = 10
MIN_BEHAV_TRIAS = 400
MIN_RT = 0.08  # 0.08  # Float (s) or None
NO_UNBIAS = False
SHUFFLE = True
COMPUTE_NEUROMETRIC = False #True if TARGET == 'signcont' else False
FORCE_POSITIVE_NEURO_SLOPES = False
# Basically, quality metric on the stability of a single unit. Should have 1 metric per neuron
QC_CRITERIA = 3/3  # 3 / 3  # In {None, 1/3, 2/3, 3/3}

BALANCED_WEIGHT = True # seems to work better with BALANCED_WEIGHT=False
HPARAM_GRID = {'alpha': np.array([0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000])}
ESTIMATORSTR = strlut[ESTIMATOR]  
if ESTIMATORSTR == 'Logistic':
    HPARAM_GRID = {'C': np.array([0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000])}
DOUBLEDIP = False
SAVE_BINNED = False  # Debugging parameter, not usually necessary
COMPUTE_NEURO_ON_EACH_FOLD = False  # if True, expect a script that is 5 times slower
ADD_TO_SAVING_PATH = '20eidV0'

# ValueErrors and NotImplementedErrors
if TARGET not in ['prior','signcont', 'pLeft','choice','feedback']:
    raise NotImplementedError('this TARGET is not supported or stable yet')

if MODEL not in list(dut.modeldispatcher.keys()):
    raise NotImplementedError('this MODEL is not supported or stable yet')

if COMPUTE_NEUROMETRIC and TARGET != 'signcont':
    raise ValueError('the target should be signcont to compute neurometric curves')

if ESTIMATORSTR == 'Logistic' and TARGET == 'choice':
    MASK_DATA = lambda x: (np.array(x)==-1)|(np.array(x)==1)
    TRANSFORM_DATA = lambda x: np.array((x+1)/2, dtype=int)
elif ESTIMATORSTR == 'Logistic' and TARGET == 'feedback':
    MASK_DATA = lambda x: (np.array(x)==-1)|(np.array(x)==1)
    TRANSFORM_DATA = lambda x: np.array((x+1)/2, dtype=int)
else:
    MASK_DATA = lambda x: (np.array(x)==np.array(x))
    TRANSFORM_DATA = lambda x: x
    
fit_metadata = {
    'criterion': SESS_CRITERION,
    'target': TARGET,
    'model_type': dut.modeldispatcher[MODEL],
    'modelfit_path': MODELFIT_PATH,
    'output_path': OUTPUT_PATH,
    'align_time': ALIGN_TIME,
    'time_window': TIME_WINDOW,
    'estimator': ESTIMATORSTR,
    'n_pseudo': N_PSEUDO,
    'min_units': MIN_UNITS,
    'min_behav_trials': MIN_BEHAV_TRIAS,
    'qc_criteria': QC_CRITERIA,
    'date': DATE,
    'shuffle': SHUFFLE,
    'no_unbias': NO_UNBIAS,
    'hyperparameter_grid': HPARAM_GRID,
    'save_binned': SAVE_BINNED,
    'balanced_weight': BALANCED_WEIGHT,
    'double_dip': DOUBLEDIP,
    'force_positive_neuro_slopes': FORCE_POSITIVE_NEURO_SLOPES,
    'compute_neurometric': COMPUTE_NEUROMETRIC
}


# %% Define helper functions for dask workers to use
def save_region_results(fit_result, pseudo_results, 
                        subject, eid, probe, region, N):#
    subjectfolder = Path(OUTPUT_PATH).joinpath(subject)
    eidfolder = subjectfolder.joinpath(eid)
    probefolder = eidfolder.joinpath(probe)
    for folder in [subjectfolder, eidfolder, probefolder]:
        if not os.path.exists(folder):
            os.mkdir(folder)
    start_tw, end_tw = TIME_WINDOW
    # fn = '_'.join([DATE, region, 
    #                'timeWindow', 
    #                str(start_tw).replace('.', '_'), 
    #                str(end_tw).replace('.', '_')]) + '.pkl'
    fn = '_'.join([DATE, region,
                   'decode', TARGET,
                   dut.modeldispatcher[MODEL] if TARGET in ['prior', 'prederr'] else 'task',
                   ESTIMATORSTR, 'align', ALIGN_TIME, 
                   str(N_PSEUDO), 'pseudosessions',
                   'timeWindow', 
                   str(start_tw).replace('.', '_'), 
                   str(end_tw).replace('.', '_')]) + '_' + ADD_TO_SAVING_PATH + '.pkl'
    fw = open(probefolder.joinpath(fn), 'wb')
    outdict = {'fit': fit_result, 'pseudosessions': pseudo_results,
               'subject': subject, 'eid': eid, 'probe': probe, 'region': region, 'N_units': N}
    pickle.dump(outdict, fw)
    fw.close()
    return probefolder.joinpath(fn)


def fit_eid(eid, sessdf):
    one = ONE()  # mode='local'
    atlas = AllenAtlas()

    estimator = ESTIMATOR #(**ESTIMATOR_KWARGS)

    subject = sessdf.xs(eid, level='eid').index[0]
    subjeids = sessdf.xs(subject, level='subject').index.unique()
    brainreg = dut.BrainRegions()
    behavior_data = mut.load_session(eid, one=one)
    pLeft_vec = np.array(behavior_data['probabilityLeft'])
    try:
        tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                                  modeltype=MODEL, beh_data=behavior_data,
                                  one=one)
    except ValueError:
        print('Model not fit.')
        tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                                  modeltype=MODEL, one=one)
    
    try:
        trialsdf = bbone.load_trials_df(eid, one=one, addtl_types=['firstMovement_times'])
        if len(trialsdf) != len(tvec):
            raise IndexError
    except IndexError:
        raise IndexError('Problem in the dimensions of dataframe of session')
    trialsdf['react_times'] = trialsdf['firstMovement_times'] - trialsdf['goCue_times']
    mask = trialsdf[ALIGN_TIME].notna()
    mask = mask & MASK_DATA(tvec)
    if NO_UNBIAS:
        mask = mask & (trialsdf.probabilityLeft != 0.5).values
    if MIN_RT is not None:
        mask = mask & (~(trialsdf.react_times < MIN_RT)).values

    nb_trialsdf = trialsdf[mask]
    msub_tvec = tvec[mask]
    msub_tvec = TRANSFORM_DATA(msub_tvec)
    

    # doubledipping
    if DOUBLEDIP:
        msub_tvec = msub_tvec - np.mean(msub_tvec)

    filenames = []
    if len(msub_tvec) <= MIN_BEHAV_TRIAS:
        return filenames

    print(f'Working on eid : {eid}')
    for probe in tqdm(sessdf.loc[subject, eid].probe, desc='Probe: ', leave=False):
        # load_spike_sorting_fast
        spikes, clusters, _ = bbone.load_spike_sorting_with_channel(eid,
                                                                    one=one,
                                                                    probe=probe,
                                                                    brain_atlas=atlas,
                                                                    dataset_types=['spikes.depths', 'spikes.amps'],
                                                                    aligned=True)

        beryl_reg = dut.remap_region(clusters[probe].atlas_id, br=brainreg)
        if QC_CRITERIA:
            metrics = pd.DataFrame.from_dict(quick_unit_metrics(spikes[probe].clusters,
                                                                spikes[probe].times,
                                                                spikes[probe].amps,
                                                                spikes[probe].depths,
                                                                cluster_ids=np.arange(beryl_reg.size)))
            try:
                metrics_verif = clusters[probe].metrics
                if beryl_reg.shape[0] == len(metrics_verif):
                    if not np.all(((metrics_verif.label - metrics.label) < 1e-10) + metrics_verif.label.isna()):
                        raise ValueError('there is a problem in the metric computations')
            except AttributeError:
                pass
            qc_pass = (metrics.label >= QC_CRITERIA).values
            if beryl_reg.shape[0] != len(qc_pass):
                raise IndexError('Shapes of metrics and number of clusters '
                                 'in regions dont match')
        else:
            qc_pass = np.ones_like(beryl_reg, dtype=bool)
        regions = np.unique(beryl_reg)
        # warnings.filterwarnings('ignore')
        for region in tqdm(regions, desc='Region: ', leave=False):
            reg_mask = beryl_reg == region
            reg_clu_ids = np.argwhere(reg_mask & qc_pass).flatten()
            N_units = len(reg_clu_ids)
            if N_units < MIN_UNITS:
                continue
            # or get_spike_count_in_bins
            if np.any(np.isnan(nb_trialsdf[ALIGN_TIME])):
                # if this happens, verify scrub of NaN values in all aign times before get_spike_counts_in_bins
                raise ValueError('this should not happen')
            intervals = np.vstack([nb_trialsdf[ALIGN_TIME] + TIME_WINDOW[0],
                                   nb_trialsdf[ALIGN_TIME] + TIME_WINDOW[1]]).T
            spikemask = np.isin(spikes[probe].clusters, reg_clu_ids)
            regspikes = spikes[probe].times[spikemask]
            regclu = spikes[probe].clusters[spikemask]
            binned, _ = get_spike_counts_in_bins(regspikes, regclu,
                                                 intervals)

            # doubledipping
            msub_binned = binned.T
            if DOUBLEDIP:
                msub_binned = binned.T - np.mean(binned.T, axis=0) # binned.T.astype(int)

            if len(msub_binned.shape) > 2:
                raise ValueError('Multiple bins are being calculated per trial,'
                                 'may be due to floating point representation error.'
                                 'Check window.')
            fit_result = dut.regress_target(msub_tvec, msub_binned, estimator,
                                            estimator_kwargs=ESTIMATOR_KWARGS,
                                            hyperparam_grid=HPARAM_GRID,
                                            save_binned=SAVE_BINNED, shuffle=SHUFFLE,
                                            balanced_weight=BALANCED_WEIGHT)

            fit_result['mask'] = mask
            fit_result['pLeft_vec'] = pLeft_vec

            # neurometric curve
            if COMPUTE_NEUROMETRIC:
                fit_result['full_neurometric'], fit_result['fold_neurometric'] = \
                    get_neurometric_parameters(fit_result, nb_trialsdf.reset_index(), one,
                                               compute_on_each_fold=COMPUTE_NEURO_ON_EACH_FOLD,
                                               force_positive_neuro_slopes=FORCE_POSITIVE_NEURO_SLOPES)
            else:
                fit_result['full_neurometric'] = None
                fit_result['fold_neurometric'] = None

            pseudo_results = []
            for _ in tqdm(range(N_PSEUDO), desc='Pseudo num: ', leave=False):
                pseudosess = generate_pseudo_session(trialsdf)
                msub_pseudo_tvec = dut.compute_target(TARGET, subject, subjeids, eid,
                                                      MODELFIT_PATH, modeltype=MODEL,
                                                      beh_data=pseudosess, one=one)[mask]
                # doubledipping
                if DOUBLEDIP:
                    msub_pseudo_tvec = msub_pseudo_tvec - np.mean(msub_pseudo_tvec)

                pseudo_result = dut.regress_target(msub_pseudo_tvec, msub_binned, estimator,
                                                   estimator_kwargs=ESTIMATOR_KWARGS,
                                                   hyperparam_grid=HPARAM_GRID,
                                                   save_binned=SAVE_BINNED, shuffle=SHUFFLE,
                                                   balanced_weight=BALANCED_WEIGHT)

                # neurometric curve
                if COMPUTE_NEUROMETRIC:
                    pseudo_result['full_neurometric'], pseudo_result['fold_neurometric'] = \
                        get_neurometric_parameters(pseudo_result, pseudosess[mask].reset_index(),
                                                   one, compute_on_each_fold=COMPUTE_NEURO_ON_EACH_FOLD,
                                                   force_positive_neuro_slopes=FORCE_POSITIVE_NEURO_SLOPES)
                else:
                    pseudo_result['full_neurometric'] = None
                    pseudo_result['fold_neurometric'] = None
                
                pseudo_result['pLeft_vec'] = pLeft_vec

                pseudo_results.append(pseudo_result)
            filenames.append(save_region_results(fit_result, pseudo_results, subject,
                                                 eid, probe, region, N_units))

    return filenames

fiteidstr = """

sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
fit_eid(eid,sessdf)

"""


In [ ]:
SLURM_DIRECTORY = os.path.join(SCRATCH,'/international-brain-lab/prior-localization/slurm/')
    

# Make top level directories
mkdir_p(job_directory)
mkdir_p(data_dir)

lizards=["LizardA","LizardB"]

for lizard in lizards:
    job_file = os.path.join(SLURM_DIRECTORY,'%s.job' % eid)

    # Create lizard directories

    with open(job_file) as fh:
        fh.writelines("#!/bin/bash\n")
        fh.writelines("#SBATCH --job-name=%s.job\n" % (eid))
        fh.writelines("#SBATCH --output=%s%s.out\n" % (SLURM_DIRECTORY,eid))
        fh.writelines("#SBATCH --error=%s%s.err\n" % (SLURM_DIRECTORY,eid))
        fh.writelines("#SBATCH --time=2-0\n")
        fh.writelines("#SBATCH --mem=4G\n")
        fh.writelines("#SBATCH --qos=normal\n")
#         fh.writelines("#SBATCH --mail-type=ALL\n")
#         fh.writelines("#SBATCH --mail-user=$USER@stanford.edu\n")
        fh.writelines("python %s%s.py %s\n" %(SLURM_DIRECTORY,eid,eid))

    os.system("sbatch %s" %job_file)

In [14]:
# 
SLURM_DIRECTORY = os.path.join(SCRATCH,'international-brain-lab/prior-localization/slurm/')
SLURM_DIRECTORY

'/scratch/users/bensonb/international-brain-lab/prior-localization/slurm/'

In [2]:
# 
SLURM_DIRECTORY = os.path.join(SCRATCH,'international-brain-lab/prior-localization/slurm/')
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')
#
sessdf_eids=np.unique(sessdf_eids)[30:32]
# sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
#             '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
#             '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
#             'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
#             '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
#             '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
#             'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
#             'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
#             '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
#             '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']
for eid in sessdf_eids:
#     fit_eid(eid,sessdf)
    
    py_file = '%s%s.py' % (SLURM_DIRECTORY,eid)
    with open(py_file) as fp:
        fp.write(filestr)
        fp.write(fiteidstr)
    
    job_file = '%s%s.job' % (SLURM_DIRECTORY,eid)
    with open(job_file) as fj:
        fj.writelines("#!/bin/bash\n")
        fj.writelines("#SBATCH --job-name=%s.job\n" % (eid))
        fj.writelines("#SBATCH --output=%s%s.out\n" % (SLURM_DIRECTORY,eid))
        fj.writelines("#SBATCH --error=%s%s.err\n" % (SLURM_DIRECTORY,eid))
        fj.writelines("#SBATCH --time=2-0\n")
        fj.writelines("#SBATCH --mem=4G\n")
        fj.writelines("#SBATCH --qos=normal\n")
#         fj.writelines("#SBATCH --mail-type=ALL\n")
#         fj.writelines("#SBATCH --mail-user=$USER@stanford.edu\n")
        fj.writelines("python %s%s.py %s\n" %(SLURM_DIRECTORY,eid,eid))

    os.system("sbatch %s" %job_file)
    

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/users/bensonb/international-brain-lab/prior-localization/slurm/15948667-747b-4702-9d53-354ac70e9119.py'

In [27]:
# fit_eid(eid,sessdf)
# Generate cluster interface and map eids to workers via dask.distributed.Client
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])

subject = sessdf.xs(eid, level='eid').index[0]
sessdf.loc[subject, eid].probe

/tmp/ipykernel_6201/1709177878.py:7: PerformanceWarning: indexing past lexsort depth may impact performance.
  sessdf.loc[subject, eid].probe


subject  eid                                 
KS022    15f742e1-1043-45c9-9504-f1e8a53c1744    probe00
         15f742e1-1043-45c9-9504-f1e8a53c1744    probe01
Name: probe, dtype: object

In [25]:
sessdf.index.nlevels 

2

In [6]:
# WAIT FOR COMPUTATION TO FINISH BEFORE MOVING ON
# %% Collate results into master dataframe and save

filenames
sessdf_eids

array(['15948667-747b-4702-9d53-354ac70e9119'], dtype=object)

In [ ]:
finished = []
for fns in tmp:
    finished.extend(fns)
#%%
indexers = ['subject', 'eid', 'probe', 'region']
indexers_neurometric = ['low_slope', 'high_slope', 'low_range', 'high_range', 'shift']
resultslist = []
for fn in finished:
    fo = open(fn, 'rb')
    result = pickle.load(fo)
    fo.close()
    tmpdict = {**{x: result[x] for x in indexers},
               'fold': -1,
               'mask': ''.join([str(item) for item in list(result['fit']['mask'].values * 1)]),
               'Rsquared_test': result['fit']['Rsquared_test_full'],
               **{f'Rsquared_test_pseudo{i}': result['pseudosessions'][i]['Rsquared_test_full']
                  for i in range(N_PSEUDO)}}
    if result['fit']['full_neurometric'] is not None \
            and np.all([result['pseudosessions'][i]['full_neurometric'] is not None for i in range(N_PSEUDO)]):
        tmpdict = {**tmpdict,
                   **{idx_neuro: result['fit']['full_neurometric'][idx_neuro]
                      for idx_neuro in indexers_neurometric},
                   **{str(idx_neuro) + f'_pseudo{i}': result['pseudosessions'][i]['full_neurometric'][idx_neuro]
                      for i in range(N_PSEUDO) for idx_neuro in indexers_neurometric}}
    resultslist.append(tmpdict)
    for kfold in range(result['fit']['nFolds']):
        tmpdict = {**{x: result[x] for x in indexers},
                   'fold': kfold,
                   'Rsquared_test': result['fit']['Rsquareds_test'][kfold],
                   'Best_regulCoef': result['fit']['best_params'][kfold],
                   **{f'Rsquared_test_pseudo{i}': result['pseudosessions'][i]['Rsquareds_test'][kfold]
                      for i in range(N_PSEUDO)},
                   }
        if result['fit']['fold_neurometric'] is not None:
            tmpdict = {**tmpdict,
                       **{idx_neuro: result['fit']['fold_neurometric'][kfold][idx_neuro]
                          for idx_neuro in indexers_neurometric}}
        if np.all([result['pseudosessions'][i]['fold_neurometric'] is not None for i in range(N_PSEUDO)]):
            tmpdict = {**tmpdict,
                       **{str(idx_neuro) + f'_pseudo{i}': result['pseudosessions'][i][
                           'fold_neurometric'][kfold][idx_neuro]
                          for i in range(N_PSEUDO) for idx_neuro in indexers_neurometric}
                       }
        resultslist.append(tmpdict)
resultsdf = pd.DataFrame(resultslist).set_index(indexers)

start_tw, end_tw = TIME_WINDOW
fn = OUTPUT_PATH + '_'.join([DATE, 'decode', TARGET,
                             dut.modeldispatcher[MODEL] if TARGET in ['prior', 'prederr'] else 'task',
                             ESTIMATORSTR, 'align', ALIGN_TIME, str(N_PSEUDO), 'pseudosessions',
                             'timeWindow', str(start_tw).replace('.', '_'), str(end_tw).replace('.', '_')])
if ADD_TO_SAVING_PATH != '':
    fn = fn + '_' + ADD_TO_SAVING_PATH
fn = fn + '.parquet'

metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
resultsdf.to_parquet(fn)
metadata_df.to_pickle(metadata_fn)

# command to close the ongoing placeholder
# client.close(); cluster.close()

# If you want to get the errors per-failure in the run:
"""
failures = [(i, x) for i, x in enumerate(filenames) if x.status == 'error']
for i, failure in failures:
print(i, failure.exception(), failure.key)
print(len(failures))
print(np.array(failures)[:,0])
print(len([(i, x) for i, x in enumerate(filenames) if x.status == 'pending']))
import traceback
tb = failure.traceback()
traceback.print_tb(tb)
"""
# You can also get the traceback from failure.traceback and print via `import traceback` and
# traceback.print_tb()